In [ ]:
# P0.3 (replikasi): pin stack era 4.x (transformers 5.0 menurunkan performa).
# torchao 0.10 tidak kompatibel dengan peft -> uninstall dulu.
!pip uninstall -y torchao
!pip install --force-reinstall --no-deps "transformers==4.46.3" "peft==0.13.2" "tokenizers==0.20.3" "huggingface-hub==0.26.5"


In [ ]:
# P0.1 (replikasi): paksa 1 GPU (DataParallel menggandakan batch -> undertrained).
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))

import sys
import torch
import transformers
import peft

assert transformers.__version__.startswith("4.46"), (
    f"transformers {transformers.__version__} bukan pin 4.46 - instalasi bermasalah!"
)
from transformers import TFPreTrainedModel  # bukti tidak ada file campur 5.0

print("python        :", sys.version)
print("torch         :", torch.__version__)
print("transformers  :", transformers.__version__)
print("peft          :", peft.__version__)
print("cuda available:", torch.cuda.is_available())
print("gpu count     :", torch.cuda.device_count())


In [ ]:
from __future__ import annotations
# =====================================================
# SUMBER KEBENARAN: src/ (disuntik oleh tools/generate_notebook.py)
# Jangan edit langsung di notebook - edit src/ lalu generate ulang.
# =====================================================

# --- src/config.py ---
"""Konfigurasi eksperimen: load dari YAML/JSON dan bantu membenamkan dict ke notebook.

Sumber kebenaran konfigurasi = file di `configs/`. Generator membaca file ini,
lalu membenamkan representasi literal dict-nya ke sel Config notebook (sel 6),
sehingga notebook Kaggle tidak butuh PyYAML.
"""

import json
from pathlib import Path
from typing import Any


def load_config(path: str | Path) -> dict[str, Any]:
    """Muat file config (.yaml/.yml/.json) menjadi dict."""
    p = Path(path)
    text = p.read_text(encoding="utf-8").strip()
    if p.suffix.lower() in (".yaml", ".yml"):
        try:
            import yaml
        except ImportError as e:
            raise ImportError(
                "PyYAML dibutuhkan untuk config YAML: pip install pyyaml"
            ) from e
        cfg = yaml.safe_load(text)
        if not isinstance(cfg, dict):
            raise ValueError(f"Config {p} harus berupa mapping YAML, bukan {type(cfg)}")
        return cfg
    return json.loads(text)


def config_repr(cfg: dict[str, Any]) -> str:
    """Representasi Python literal (json.dumps) untuk dibenamkan di sel notebook."""
    return json.dumps(cfg, indent=4, ensure_ascii=False)


def config_snippet(cfg: dict[str, Any], var_name: str = "CONFIG") -> str:
    """Source untuk sel Config: `CONFIG = {...}`."""
    return f"{var_name} = {config_repr(cfg)}"

# --- src/data.py ---
"""Data: loader dataset fleksibel (path mount Kaggle CLI 2.x vs lama), split, dataset PyTorch.

Sumber kebenaran loading data untuk semua eksperimen. Sel notebook menyuntik source
fungsi-fungsi di sini (via generator) sehingga notebook tetap self-contained di Kaggle.
"""

import os
from typing import Any

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset

COL_TEXT = "text_bert"
COL_LABEL = "label"
CSV_NAME = "data_preprocessed_with_emoticon.csv"


def find_dataset_csv() -> str:
    """Cari CSV dataset di /kaggle/input (path mount berubah antara CLI 2.x dan lama).

    - CLI 2.x:  /kaggle/input/datasets/<owner>/<slug>/...
    - Skema lama: /kaggle/input/<slug>/...
    Tidak ada hardcode path: cari dari daftar file ter-mount.
    """
    mounted = []
    for root, _dirs, files in os.walk("/kaggle/input"):
        for f in files:
            if f == CSV_NAME:
                mounted.append(os.path.join(root, f))
    if not mounted:
        raise FileNotFoundError(
            f"Dataset '{CSV_NAME}' tidak ditemukan di /kaggle/input. "
            "Cek dataset_sources di kernel-metadata.json."
        )
    return mounted[0]


def load_dataframe() -> pd.DataFrame:
    """Muat CSV dataset dengan validasi kolom BERT eksplisit (text_bert)."""
    path = find_dataset_csv()
    print("CSV ditemukan di:", path)
    df = pd.read_csv(path)
    if COL_TEXT not in df.columns:
        raise ValueError(
            f"Kolom '{COL_TEXT}' tidak ditemukan di CSV. Kolom tersedia: {df.columns.tolist()}"
        )
    df[COL_TEXT] = df[COL_TEXT].fillna("").astype(str)
    print(f"Kolom BERT terpilih: {COL_TEXT} | Total baris: {len(df)}")
    return df


def split_data(
    df: pd.DataFrame,
    test_size: float = 0.2,
    val_size: float = 0.1,
    random_state: int = 42,
) -> dict[str, np.ndarray]:
    """Split 80:20 (test) lalu 90:10 (val) — protokol konsisten semua eksperimen."""
    train_df, test_df = train_test_split(
        df, test_size=test_size, random_state=random_state, stratify=df[COL_LABEL]
    )
    X_train = train_df[COL_TEXT].values
    X_test = test_df[COL_TEXT].values
    y_train = train_df[COL_LABEL].values
    y_test = test_df[COL_LABEL].values

    X_train_final, X_val, y_train_final, y_val = train_test_split(
        X_train, y_train, test_size=val_size, stratify=y_train, random_state=random_state
    )
    return {
        "X_train": X_train_final,
        "X_val": X_val,
        "X_test": X_test,
        "y_train": y_train_final,
        "y_val": y_val,
        "y_test": y_test,
    }


class SentimenDataset(Dataset):
    """Dataset PyTorch numpy-friendly (menerima numpy array & pandas Series)."""

    def __init__(
        self,
        texts: Any,
        labels: Any,
        tokenizer,
        max_length: int = 128,
    ):
        self.texts = texts.values if isinstance(texts, pd.Series) else np.array(texts)
        self.labels = labels.values if isinstance(labels, pd.Series) else np.array(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        text = str(self.texts[idx])
        label = int(self.labels[idx])
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "labels": torch.tensor(label, dtype=torch.long),
        }

# --- src/model.py ---
"""Model: builder IndoBERTweet-LoRA (sumber kebenaran tunggal)."""

from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    PreTrainedTokenizerFast,
)

MODEL_NAME = "indolem/indobertweet-base-uncased"
ID2LABEL = {0: "negatif", 1: "netral", 2: "positif"}
LABEL2ID = {"negatif": 0, "netral": 1, "positif": 2}


def build_indobertweet_lora(
    dropout: float = 0.3,
    r: int = 16,
    lora_alpha: int = 32,
    num_labels: int = 3,
):
    """Bangun model IndoBERTweet + LoRA (r, alpha, dropout) dengan classifier baru."""
    config = AutoConfig.from_pretrained(
        MODEL_NAME,
        num_labels=num_labels,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
        hidden_dropout_prob=dropout,
        attention_probs_dropout_prob=dropout,
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, config=config, ignore_mismatched_sizes=True
    )
    model.config.id2label = ID2LABEL
    model.config.label2id = LABEL2ID

    lora_config = LoraConfig(
        r=r,
        lora_alpha=lora_alpha,
        target_modules=["query", "value"],
        lora_dropout=dropout,
        bias="none",
        task_type=TaskType.SEQ_CLS,
        modules_to_save=["classifier"],
    )
    return get_peft_model(model, lora_config)


def load_tokenizer() -> PreTrainedTokenizerFast:
    from transformers import AutoTokenizer

    return AutoTokenizer.from_pretrained(MODEL_NAME)

# --- src/metrics.py ---
"""Metrik evaluasi: compute_metrics (HF) + softmax numpy (untuk simpan probabilitas)."""

import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

LABEL_NAMES = ["negatif", "netral", "positif"]


def compute_metrics(eval_pred):
    """Metrik untuk HF Trainer (average='macro', zero_division=0)."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": acc,
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1,
    }


def softmax_np(logits: np.ndarray) -> np.ndarray:
    """Softmax stabil (numerik) di axis=1."""
    z = logits - logits.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def prediction_frame(
    texts,
    y_true,
    logits: np.ndarray,
    prob_cols=("prob_negatif", "prob_netral", "prob_positif"),
):
    """DataFrame per-sampel: teks, label aktual/prediksi, probabilitas per kelas."""
    import pandas as pd

    y_pred = np.argmax(logits, axis=1)
    P = softmax_np(logits)
    return pd.DataFrame(
        {
            "text": pd.Series(texts),
            "label_aktual": pd.Series(y_true),
            "label_prediksi": pd.Series(y_pred),
            prob_cols[0]: P[:, 0],
            prob_cols[1]: P[:, 1],
            prob_cols[2]: P[:, 2],
        }
    )

# --- src/trainer_factory.py ---
"""Trainer Factory: satu fungsi untuk membangun Trainer dengan loss yang dipilih.

Sumber kebenaran tunggal untuk:
- "cross_entropy" -> Trainer standar HF
- "weighted_ce"   -> WeightedTrainer (CrossEntropyLoss(weight=class_weight))
- "focal"         -> FocalLossTrainer (gamma, alpha=class_weight)

Sel notebook menyuntik source file ini, sehingga tidak ada duplikasi kode Trainer
di antar-notebook (akar bug 'trainer_best' & compute_loss ganda di masa lalu).
"""

import torch
import torch.nn.functional as F
from torch import nn
from transformers import Trainer


class WeightedTrainer(Trainer):
    """CrossEntropyLoss dengan bobot kelas."""

    def __init__(self, class_weight=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weight = (
            torch.tensor(class_weight, dtype=torch.float) if class_weight is not None else None
        )

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        if self.class_weight is not None:
            loss_fct = nn.CrossEntropyLoss(weight=self.class_weight.to(logits.device))
        else:
            loss_fct = nn.CrossEntropyLoss()
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


class FocalLossTrainer(Trainer):
    """Focal Loss: FL(p_t) = -alpha_t (1-p_t)^gamma log(p_t), dengan alpha opsional."""

    def __init__(self, gamma=2.0, class_weight=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.gamma = gamma
        self.class_weight = (
            torch.tensor(class_weight, dtype=torch.float) if class_weight is not None else None
        )

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        probs = torch.softmax(logits, dim=-1)
        pt = probs.gather(1, labels.unsqueeze(1)).squeeze(1)
        focal = -(1.0 - pt) ** self.gamma * torch.log(pt.clamp(min=1e-8))
        if self.class_weight is not None:
            alpha_t = self.class_weight.to(logits.device).gather(0, labels)
            focal = focal * alpha_t
        loss = focal.mean()
        return (loss, outputs) if return_outputs else loss


def build_trainer(
    loss: str = "cross_entropy",
    class_weight=None,
    gamma: float = 2.0,
    **trainer_kwargs,
) -> Trainer:
    """Bangun Trainer sesuai strategi loss.

    Contoh:
        build_trainer(loss="weighted_ce", class_weight=[0.75, 1.32, 1.03], args=args, ...)
        build_trainer(loss="focal", gamma=2.0, class_weight=[...], args=args, ...)
    """
    if loss == "cross_entropy":
        trainer_cls = Trainer
    elif loss == "weighted_ce":
        trainer_cls = WeightedTrainer
    elif loss == "focal":
        trainer_cls = FocalLossTrainer
    else:
        raise ValueError(f"Loss tidak dikenal: {loss} (pilihan: cross_entropy, weighted_ce, focal)")

    if loss == "weighted_ce":
        return WeightedTrainer(class_weight=class_weight, **trainer_kwargs)
    if loss == "focal":
        return FocalLossTrainer(gamma=gamma, class_weight=class_weight, **trainer_kwargs)
    return Trainer(**trainer_kwargs)

# --- src/summary.py ---
"""Auto Experiment Summary: menulis metadata run (config, commit, dataset MD5) ke file & stdout."""

import hashlib
import json
import subprocess
from pathlib import Path
from typing import Any


def git_commit_short() -> str:
    """Hash commit git saat ini (atau 'unknown' bila bukan repo git)."""
    try:
        out = subprocess.run(
            ["git", "rev-parse", "--short", "HEAD"],
            capture_output=True,
            text=True,
            timeout=10,
        )
        return out.stdout.strip() or "unknown"
    except Exception:
        return "unknown"


def file_md5(path: str | Path) -> str:
    """MD5 file (untuk audit data lineage)."""
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()


def experiment_summary(
    exp_id: str,
    config: dict[str, Any],
    metrics: dict[str, Any],
    csv_path: str | None = None,
    dataset_path: str | None = None,
    out_path: str | None = None,
) -> dict[str, Any]:
    """Buat dict ringkasan + tulis file JSON (opsional) + cetak ke stdout.

    Metrics contoh: {"accuracy": ..., "f1_macro": ..., "netral_f1": ...}
    """
    summary = {
        "experiment": exp_id,
        "commit": git_commit_short(),
        "config": config,
        "metrics": metrics,
    }
    if csv_path:
        summary["prediction_csv"] = csv_path
    if dataset_path:
        summary["dataset_md5"] = file_md5(dataset_path)

    if out_path:
        Path(out_path).parent.mkdir(parents=True, exist_ok=True)
        Path(out_path).write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")

    print("=" * 60)
    print("AUTO EXPERIMENT SUMMARY")
    print("=" * 60)
    print(f"Experiment : {exp_id}")
    print(f"Commit     : {summary.get('commit')}")
    if dataset_path:
        print(f"Dataset MD5: {summary['dataset_md5']}")
    print(f"Config     : {json.dumps(config, ensure_ascii=False)}")
    print(f"Metrics    : {json.dumps(metrics, ensure_ascii=False)}")
    if out_path:
        print(f"Summary    : {out_path}")
    return summary

In [ ]:
# =====================================================
# SET SEED
# =====================================================
seed = 42
set_seed(seed)
np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
print("GPU tersedia:", torch.cuda.is_available())


In [ ]:
# =====================================================
# DATASET (loader fleksibel + validasi text_bert)
# =====================================================
df = load_dataframe()
split = split_data(df, test_size=0.2, val_size=0.1, random_state=seed)

tokenizer = load_tokenizer()
train_dataset = SentimenDataset(split["X_train"], split["y_train"], tokenizer, max_length=CONFIG["max_length"])
val_dataset = SentimenDataset(split["X_val"], split["y_val"], tokenizer, max_length=CONFIG["max_length"])
test_dataset = SentimenDataset(split["X_test"], split["y_test"], tokenizer, max_length=CONFIG["max_length"])
print(f"Train {len(train_dataset)} | Val {len(val_dataset)} | Test {len(test_dataset)}")


In [ ]:
# =====================================================
# CONFIG (dibenamkan dari configs/exp_p1_weightedce.yaml)
# =====================================================
CONFIG = {
    "exp_id": "exp_p1_weightedce",
    "title": "Thesis LoRA Weighted CE",
    "description": "P1 - Class-Weighted CrossEntropy (0.75/1.32/1.03), label corrected",
    "loss": "weighted_ce",
    "class_weight": [
        0.75,
        1.32,
        1.03
    ],
    "gamma": 2.0,
    "params": {
        "learning_rate": 0.0002,
        "batch_size": 16,
        "epochs": 5,
        "weight_decay": 0.01,
        "dropout": 0.3,
        "lora_r": 16,
        "lora_alpha": 32,
        "max_length": 128
    },
    "dataset_sources": [
        "emanuelembuaijdak/thesis-indobert-processed-data"
    ]
}
print(json.dumps(CONFIG, indent=2))

In [ ]:
# =====================================================
# MODEL (build LoRA dari CONFIG)
# =====================================================
model = build_indobertweet_lora(
    dropout=CONFIG["dropout"],
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
)
model.print_trainable_parameters()


In [ ]:
# =====================================================
# TRAINING (trainer_factory: loss=weighted_ce)
# =====================================================
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results_exp_p1_weightedce",
    learning_rate=0.0002,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=50,
    report_to="none",
    save_total_limit=1,
)

trainer = build_trainer(
    loss='weighted_ce',
    class_weight=[0.75, 1.32, 1.03],
    gamma=2.0,
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)
trainer.train()
eval_result = trainer.evaluate()
print("Hasil validation:", eval_result)

# --- Sanity check (P0): deteksi collapse ---
preds_val = trainer.predict(val_dataset)
y_val_pred = np.argmax(preds_val.predictions, axis=1)
maj = pd.Series(split["y_val"]).mode()[0]
p_maj = float((split["y_val"] == maj).mean())
f1_maj = (2 * p_maj / (1 + p_maj)) / 3
print("Distribusi prediksi val :", pd.Series(y_val_pred).value_counts().sort_index().to_dict())
print("Baseline mayoritas val  : acc=" + str(round(p_maj, 4)) + " macro_f1=" + str(round(f1_maj, 4)))
status = "COLLAPSE" if eval_result["eval_f1_macro"] <= f1_maj + 1e-6 else "OK"
print("STATUS:", status)


In [ ]:
# =====================================================
# EVALUASI TEST + SIMPAN PROBABILITAS
# =====================================================
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support

preds_test = trainer.predict(test_dataset)
logits = preds_test.predictions
y_pred_test = np.argmax(logits, axis=1)
P = softmax_np(logits)

print(classification_report(split["y_test"], y_pred_test, target_names=LABEL_NAMES, zero_division=0))
print("Distribusi prediksi:", pd.Series(y_pred_test).value_counts().sort_index().to_dict())
print("Distribusi aktual  :", pd.Series(split["y_test"]).value_counts().sort_index().to_dict())

hasil = prediction_frame(split["X_test"], split["y_test"], logits)
fname = "exp_p1_weightedce_test.csv"
hasil.to_csv(fname, index=False)
print("Tersimpan:", fname)

precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
    split["y_test"], y_pred_test, average="macro", zero_division=0
)
accuracy = accuracy_score(split["y_test"], y_pred_test)

cm = confusion_matrix(split["y_test"], y_pred_test)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES)
plt.title("Confusion Matrix - exp_p1_weightedce")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


In [ ]:
# =====================================================
# SAVE ARTIFACT + AUTO EXPERIMENT SUMMARY
# =====================================================
import json

metrics = {
    "accuracy": accuracy,
    "precision_macro": precision_macro,
    "recall_macro": recall_macro,
    "f1_macro": f1_macro,
}
summary = experiment_summary(
    exp_id='exp_p1_weightedce',
    config=CONFIG,
    metrics=metrics,
    csv_path="exp_p1_weightedce_test.csv",
    out_path="exp_p1_weightedce_summary.json",
)
